In [ ]:
# truth_subspace_and_steering.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional
import numpy as np
from sklearn.decomposition import PCA

# =========================
# Utils
# =========================

def normalize_vec(v: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """Normalize last-dim vectors to unit norm."""
    return v / (v.norm(dim=-1, keepdim=True) + eps)

def orthonormalize_columns(U: torch.Tensor) -> torch.Tensor:
    """QR-based orthonormalization for column vectors. U: (d, k) -> Q (d, k)"""
    Q, _ = torch.linalg.qr(U, mode='reduced')
    return Q

def project_out(v: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    """
    Remove component of v along unit vector u.
    v: (..., d), u: (d,) assumed unit
    """
    coeff = torch.matmul(v, u)  # (...,)
    return v - coeff.unsqueeze(-1) * u

# =========================
# Build truth subspace U = [r0 | U_rest]
# =========================



# =========================
# Optional: benign subspace (as a regularizer)
# =========================

def build_benign_subspace(
    Hb: torch.Tensor,            # (Nb, d) 良性样本隐藏态
    kb: int = 32,
    center: bool = True
) -> torch.Tensor:
    """
    返回 B_basis: (d, kb) 良性子空间基（列正交）
    """
    if center:
        X = Hb - Hb.mean(dim=0, keepdim=True)
    else:
        X = Hb
    X_np = X.detach().cpu().numpy()
    kb = max(1, min(kb, X.shape[1]))
    pca = PCA(n_components=kb, svd_solver="auto")
    pca.fit(X_np)
    B_basis = torch.from_numpy(pca.components_.T).to(Hb)
    B_basis = orthonormalize_columns(B_basis)
    return B_basis  # (d, kb)

# =========================
# Steering Adapter
# =========================

class SteeringAdapter(nn.Module):
    """
    s(h) = alpha(h)*r0 + U_rest @ beta(h)
    g(h) in [0,1], final h' = h + g * s(h)
    注意：这里 U 传入时可以是完整 [r0 | U_rest]，但实现上我们单独用 r0 与 U_rest 更清晰。
    """
    def __init__(self, input_dim: int, hidden_dim: int, k_rest: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # Sigmoid 在 forward 里做
        )
        self.scale = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.mix = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, k_rest)
        )
        self.gamma = nn.Parameter(torch.tensor(1.0))  # 全局强度

    def forward(
        self,
        h: torch.Tensor,     # (B, d)
        r0: torch.Tensor,    # (d,)   unit
        U_rest: torch.Tensor # (d, k_rest) columns orthonormal; can be empty if k_rest=0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        B, d = h.shape
        assert r0.shape == (d,), f"r0 must be (d,), got {r0.shape}"
        k_rest = U_rest.shape[1] if U_rest.ndim == 2 else 0

        g_logit = self.gate(h)          # (B,1)
        g = torch.sigmoid(g_logit)      # (B,1)

        alpha = self.scale(h)           # (B,1)
        beta  = self.mix(h) if k_rest > 0 else torch.zeros(B, 0, device=h.device, dtype=h.dtype)  # (B,k_rest)

        s_r0 = alpha * r0.unsqueeze(0)  # (B,d)
        s_U  = beta @ U_rest.T if k_rest > 0 else torch.zeros_like(s_r0)  # (B,d)

        s = self.gamma * (s_r0 + s_U)   # (B,d)
        h_prime = h + g * s             # (B,d)
        return g, alpha, beta, s, h_prime

# =========================
# Losses
# =========================

def cosine_alignment_loss(x: torch.Tensor, y: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    1 - cosine(x, y). x,y: (B,d)
    mask: (B,) in {0,1}, optional
    """
    x_n = normalize_vec(x)
    y_n = normalize_vec(y)
    cos = (x_n * y_n).sum(dim=-1)  # (B,)
    loss = 1.0 - cos
    if mask is not None:
        loss = (loss * mask).sum() / (mask.sum() + 1e-12)
    else:
        loss = loss.mean()
    return loss

def gating_bce_loss(g: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    BCE for gate. g: (B,1) in [0,1]; y: (B,) in {0,1}
    """
    y = y.float().unsqueeze(-1)  # (B,1)
    return F.binary_cross_entropy(g, y)

def magnitude_penalty(s: torch.Tensor, mask: Optional[torch.Tensor] = None, weight: float = 1.0) -> torch.Tensor:
    """
    ||s||^2 penalty. s: (B,d)
    mask: (B,), apply only where mask==1
    """
    per = (s**2).sum(dim=-1)  # (B,)
    if mask is not None:
        per = per * mask
        return weight * per.sum() / (mask.sum() + 1e-12)
    return weight * per.mean()

def benign_orth_penalty(s: torch.Tensor, B_basis: torch.Tensor, mask: Optional[torch.Tensor] = None, weight: float = 1.0) -> torch.Tensor:
    """
    Penalize energy of s in benign subspace: ||s_B||^2, where s_B = s @ B
    s: (B,d), B_basis: (d, kb)
    """
    if B_basis is None or B_basis.numel() == 0:
        return torch.tensor(0.0, device=s.device, dtype=s.dtype)
    s_B = s @ B_basis  # (B, kb)
    per = (s_B**2).sum(dim=-1)  # (B,)
    if mask is not None:
        per = per * mask
        return weight * per.sum() / (mask.sum() + 1e-12)
    return weight * per.mean()

# =========================
# Example training skeleton
# =========================

def example_training():
    """
    用随机数据演示完整流程（请用你的真实隐藏态替换）。
    设:
      - 每个问题有一个正确隐藏态 Hc[i] 与 K=3 个错误隐藏态 Hi[i, k]
      - 输入 h 这里用 (Hc + mean(Hi))/2 作为示例的“查询/问题隐藏态”，实际请用你的 query/encoder 输出
      - y=1 表示需要纠偏（有幻觉倾向），y=0 表示良性（不该动）
    """
    torch.manual_seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    M, d, K = 1024, 512, 3
    k_total = 32

    # ---- 替换为你的真实数据 ----
    Hc = torch.randn(M, d, device=device)             # 正确回复隐藏态
    Hi = torch.randn(M, K, d, device=device)          # 多个错误回复隐藏态
    Hb = torch.randn(M, d, device=device)             # 良性样本（可选，用于 B 子空间）

    # 构建 truth 子空间
    r0, U = build_truth_subspace(Hc, Hi, k=k_total, device=device, dtype=Hc.dtype)  # r0:(d,), U:(d,k)
    U_rest = U[:, 1:] if U.shape[1] > 1 else torch.zeros(d, 0, device=device, dtype=Hc.dtype)

    # 可选：良性子空间（用于正则）
    B_basis = build_benign_subspace(Hb, kb=16)

    # 构造训练输入 h（实际请用你的“问题/查询隐藏态”）
    Hi_mean = Hi.mean(dim=1)                # (M, d)
    Hq = 0.5 * (Hc + Hi_mean)               # (M, d) 仅示例

    # 目标方向（逐样本）：Δ_i = Hc - mean(Hi)
    Delta = Hc - Hi_mean                    # (M, d)

    # 构造简易标签 y：一半作为“需要纠偏”（y=1），一半作为良性（y=0）
    y = torch.zeros(M, device=device)
    y[: M // 2] = 1.0
    perm = torch.randperm(M, device=device)
    Hq, Delta, y = Hq[perm], Delta[perm], y[perm]

    # 模型
    model = SteeringAdapter(input_dim=d, hidden_dim=512, k_rest=U_rest.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

    # 超参
    lambda_gate = 1.0
    lambda_dir  = 1.0
    lambda_mag  = 0.2
    lambda_benign_orth = 0.1
    batch_size = 64
    steps = 200

    for step in range(steps):
        idx = torch.randint(0, M, (batch_size,), device=device)
        h_batch    = Hq[idx]            # (B,d)
        delta_batch= Delta[idx]         # (B,d)
        y_batch    = y[idx]             # (B,)

        g, alpha, beta, s, h_prime = model(h_batch, r0, U_rest)

        # 损失：
        # 1) 门控 BCE：希望 y=1 的开、y=0 的关
        loss_gate = gating_bce_loss(g, y_batch)

        # 2) 有害样本（y=1）上，对齐目标方向：s 与 Δ_i 对齐（也可只对齐 r0）
        harm_mask = y_batch
        loss_dir = cosine_alignment_loss(s, delta_batch, mask=harm_mask)

        # 3) 良性样本（y=0）上，抑制改动能量
        benign_mask = 1.0 - y_batch
        loss_mag = magnitude_penalty(s, mask=benign_mask, weight=1.0)

        # 4) 可选：避免 s 掉进“良性子空间” B
        loss_orth = benign_orth_penalty(s, B_basis, mask=benign_mask, weight=1.0)

        loss = (lambda_gate * loss_gate +
                lambda_dir  * loss_dir  +
                lambda_mag  * loss_mag  +
                lambda_benign_orth * loss_orth)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        if (step + 1) % 20 == 0:
            with torch.no_grad():
                # 打印一些监控量
                avg_g_harm   = g[harm_mask.bool()].mean().item() if harm_mask.sum() > 0 else 0.0
                avg_g_benign = g[benign_mask.bool()].mean().item() if benign_mask.sum() > 0 else 0.0
                print(f"step {step+1:4d} | loss {loss.item():.4f} | gate(harm) {avg_g_harm:.3f} | gate(benign) {avg_g_benign:.3f}")

if __name__ == "__main__":
    example_training()


step   20 | loss 1.6829 | gate(harm) 0.516 | gate(benign) 0.482
step   40 | loss 1.5447 | gate(harm) 0.540 | gate(benign) 0.471
step   60 | loss 1.3916 | gate(harm) 0.576 | gate(benign) 0.475
step   80 | loss 1.3197 | gate(harm) 0.572 | gate(benign) 0.394
step  100 | loss 1.2239 | gate(harm) 0.620 | gate(benign) 0.402
step  120 | loss 1.1508 | gate(harm) 0.646 | gate(benign) 0.345
step  140 | loss 1.1315 | gate(harm) 0.649 | gate(benign) 0.358
step  160 | loss 1.0595 | gate(harm) 0.706 | gate(benign) 0.342
step  180 | loss 1.0385 | gate(harm) 0.695 | gate(benign) 0.301
step  200 | loss 0.9705 | gate(harm) 0.748 | gate(benign) 0.253


In [3]:
import torch
import datasets
import pickle
[train_ds]=pickle.load(open("train.pkl",'rb'))

In [4]:
train_ds

Dataset({
    features: ['correct_answers', 'incorrect_answers', 'question', 'template_q', 'category', 'y_win_layer18', 'y_lose_layer18', 'hc_layer18', 'hi_layer18', 'y_win_layer20', 'y_lose_layer20', 'hc_layer20', 'hi_layer20', 'y_win_layer22', 'y_lose_layer22', 'hc_layer22', 'hi_layer22'],
    num_rows: 408
})

In [5]:
aaa=next(iter(train_ds))

In [6]:
aaa

{'y_win_layer20': tensor([[ 0.0369,  1.1846, -3.4746,  ...,  0.1641, -2.1387,  0.7451]]),
 'y_lose_layer20': tensor([[-0.7686, -3.6113, -1.9209,  ...,  3.3320,  0.0520,  3.2832]])}

In [9]:
[ds]=pickle.load(open("ds.pkl",'rb'))

In [23]:
train_ds_new=ds['train']

In [69]:
aaa="aec"

In [73]:
"".join(sorted(aaa))

'ace'

In [17]:
hc=bbb['hc_layer20']

In [19]:
hc[0]

[-0.04254150390625,
 0.78173828125,
 -3.15234375,
 -3.373046875,
 -1.6318359375,
 -1.37109375,
 2.91015625,
 0.62060546875,
 0.045074462890625,
 0.57958984375,
 1.55078125,
 -0.3974609375,
 0.354248046875,
 -2.4140625,
 1.4306640625,
 4.2109375,
 -2.599609375,
 -0.76806640625,
 -0.72216796875,
 -0.0311737060546875,
 0.7568359375,
 -0.1793212890625,
 -1.0283203125,
 -0.87255859375,
 1.54296875,
 -1.173828125,
 6.05078125,
 -0.52099609375,
 1.19140625,
 0.2208251953125,
 0.66259765625,
 0.74267578125,
 2.517578125,
 2.2578125,
 -0.71630859375,
 -2.83203125,
 -0.8837890625,
 1.1494140625,
 -1.3427734375,
 1.03515625,
 -3.212890625,
 -0.166015625,
 0.64501953125,
 2.15234375,
 -1.255859375,
 0.2330322265625,
 0.1214599609375,
 -2.017578125,
 -1.3671875,
 -1.228515625,
 -0.90625,
 1.4853515625,
 -0.63330078125,
 1.8828125,
 0.399169921875,
 -1.2578125,
 -0.98388671875,
 -2.15625,
 -2.310546875,
 0.19921875,
 -0.626953125,
 1.8330078125,
 -1.0654296875,
 -0.11016845703125,
 0.23291015625,
 -

In [24]:
train_ds_new

Dataset({
    features: ['correct_answers', 'incorrect_answers', 'question', 'template_q', 'category', 'y_win_layer18', 'y_lose_layer18', 'hc_layer18', 'hi_layer18', 'y_win_layer20', 'y_lose_layer20', 'hc_layer20', 'hi_layer20', 'y_win_layer22', 'y_lose_layer22', 'hc_layer22', 'hi_layer22'],
    num_rows: 408
})

In [35]:
hc_list=[]
hi_list=[]
y_win_set=[]
for example in train_ds_new:
    hc=example["hc_layer20"]
    hi=example["hc_layer20"]
    y_win=example["y_win_layer20"]
    hc_list.append(hc[0])
    hi_list.append(hi[0])
    y_win_set.append(y_win)

In [32]:
hc_list=torch.cat(hc_list)
hi_list=torch.cat(hi_list)

TypeError: expected Tensor as element 0 in argument 0, but got list

In [34]:
len(hc_list)

408

In [38]:
len(y_win_set[0])

1

In [ ]:
train_ds_new.set_format(type='torch')

In [52]:
train_ds_new

Dataset({
    features: ['correct_answers', 'incorrect_answers', 'question', 'template_q', 'category', 'y_win_layer18', 'y_lose_layer18', 'hc_layer18', 'hi_layer18', 'y_win_layer20', 'y_lose_layer20', 'hc_layer20', 'hi_layer20', 'y_win_layer22', 'y_lose_layer22', 'hc_layer22', 'hi_layer22'],
    num_rows: 408
})

In [53]:
layers=[20]
y_win_set = [[] for _ in range(len(layers))]#这个是正确答案减去错误答案激活的差值。
y_lose_set = [[] for _ in range(len(layers))]#这个是question的激活
hc_set = [[] for _ in range(len(layers))]
hi_set = [[] for _ in range(len(layers))]
for example in train_ds_new:
    for idx, layer in enumerate(layers):
        y_win = example[f"y_win_layer{layer}"]#[1,3584]
        y_lose = example[f"y_lose_layer{layer}"]#[1,3584]
        hc=example[f"hc_layer{layer}"]
        hi=example[f'hi_layer{layer}']
        # y_win_pair = y_win.repeat(1, y_lose.shape[0]).reshape(-1, y_win.shape[1])#这个就是让y_win能和y_lose配对, shape[1,3584]
        # y_lose_pair = y_lose.tile((y_win.shape[0], 1))#扩展 lose，让它重复配对所有 win。shape[1,3584]
        y_win_set[idx].append(y_win)
        y_lose_set[idx].append(y_lose)
        hc_set[idx].append(hc)
        hi_set[idx].append(hi)

In [54]:
y_win_set = [torch.cat(y_win_per_layer) for y_win_per_layer in y_win_set]#每个元素是每一层的激活几何。然后每个元素的shape是[408,3584]，408是所有prompt的数量，3584是hidden size
y_lose_set = [torch.cat(y_lose_per_layer) for y_lose_per_layer in y_lose_set]#这个也同理
hc_set=[torch.cat(hc_per_layer) for hc_per_layer in hc_set]
hi_set=[torch.cat(hi_per_layer) for hi_per_layer in hi_set]

In [57]:
hc_set[0].shape

torch.Size([408, 3584])

In [58]:
hi_set[0].shape

torch.Size([408, 3584])

In [61]:
def build_truth_subspace(
    Hc: torch.Tensor,               # (M, d) 正确回复隐藏态
    Hi: torch.Tensor,               # (M, d) 或 (M, K, d) 错误回复隐藏态
    k: int = 32,                    # 子空间维数（含 r0）
    center_residual: bool = True,   # 是否对残差做去均值
    device: Optional[torch.device] = None,
    dtype: Optional[torch.dtype] = None
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    返回:
      r0: (d,) 单位向量，全局真相方向
      U:  (d, k) 列为正交基，第一列是 r0
    说明:
      - 若 Hi 是 (M, K, d)，会先对 K 个负例取均值。
      - 在残差上做 PCA，取前 k-1 个方向，最后与 r0 拼接并正交化。
    """
    assert Hc.ndim == 2, "Hc must be (M, d)"
    M, d = Hc.shape

    if Hi.ndim == 3:
        Hi_mean = Hi.mean(dim=1)  # (M, d)
    elif Hi.ndim == 2:
        Hi_mean = Hi
    else:
        raise ValueError("Hi must be (M, d) or (M, K, d)")

    if device is None:
        device = Hc.device
    if dtype is None:
        dtype = Hc.dtype

    # 1) 差分与全局真相方向 r0
    Hc=Hc.to(device)
    Hi_mean=Hi_mean.to(device)
    Delta = Hc - Hi_mean                 # (M, d)
    delta_mean = Delta.mean(dim=0)       # (d,)
    r0 = delta_mean / (delta_mean.norm() + 1e-12)  # (d,)

    # 2) 去 r0 分量后的残差
    R = project_out(Delta, r0)           # (M, d)
    if center_residual:
        R = R - R.mean(dim=0, keepdim=True)

    # 3) 在残差上做 PCA，取 k-1 个方向（如果 k=1 则只有 r0）
    k = max(1, min(k, d))
    k_rest = max(0, k - 1)

    if k_rest > 0:
        R_np = R.detach().cpu().numpy()
        pca = PCA(n_components=k_rest, svd_solver="auto")
        pca.fit(R_np)
        U_rest = torch.from_numpy(pca.components_.T).to(device=device, dtype=dtype)  # (d, k-1)

        # 保险：确保 U_rest 与 r0 正交，且列正交单位
        # 先把 r0 分量去掉再 QR
        U_rest = U_rest - r0[:, None] * (r0 @ U_rest)
        U_rest = orthonormalize_columns(U_rest)
        U = torch.cat([r0[:, None], U_rest], dim=1)  # (d, k)
    else:
        U = r0[:, None]

    # 再做一次轻微的正交化（保持 r0 不变）
    # 用 Gram-Schmidt 对其余列做正交，r0 作为第一列固定
    if U.shape[1] > 1:
        U2 = U.clone()
        U2[:, 0] = r0
        for j in range(1, U2.shape[1]):
            v = U2[:, j]
            v = project_out(v, r0)
            # 与之前列正交
            for i in range(1, j):
                vi = U2[:, i]
                v = v - vi * (vi @ v)
            v = v / (v.norm() + 1e-12)
            U2[:, j] = v
        U = U2

    return r0, U  # r0:(d,), U:(d,k)
r0, U = build_truth_subspace(hc_set[0], hi_set[0], k=8, device=torch.device("cuda"))  # r0:(d,), U:(d,k)

In [68]:
U_rest = U[:, 1:] if U.shape[1] > 1 else torch.zeros(d, 0, device=device, dtype=Hc.dtype)

In [1]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B", device_map="cpu"
)
# Llama 模型在 HF 里通常是 model.model.layers
print("blocks =", len(model.model.layers))


/hpc2hdd/home/hwang574/miniconda3/envs/alphasteer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files:   0%|          | 0/2 [05:31<?, ?it/s]


KeyboardInterrupt: 